# Gesture–Speech Alignment: Local Analysis Notebook

This is the **local-run** version of the notebook, for running on your own machine (Windows/venv) instead of Google Colab.

**Before running this notebook, make sure you've already done (in a terminal, in this project folder):**
```
venv\Scripts\activate
pip install -r requirements.txt
pip install notebook
jupyter notebook notebook\analysis_local.ipynb
```

**Requirements checked/used by this notebook:**
- `ffmpeg` on PATH (you installed this via `winget install ffmpeg` — close/reopen your terminal after installing so PATH picks it up, and run `ffmpeg -version` to confirm before starting)
- A JS runtime for yt-dlp (Deno recommended) — YouTube requires this to solve a signature challenge before downloads will work
- Optionally, cookies from a logged-in browser, if yt-dlp reports `Sign in to confirm you're not a bot`. Running locally lets you use `--cookies-from-browser chrome` directly — **more secure than the Colab route**, since no cookie file is ever written to disk; yt-dlp reads straight from Chrome's local (encrypted) cookie store at runtime.

**Fixes already baked into `src/`:**
- `download_clips.py` downloads from YouTube (ted.com's own extractor is currently broken upstream), explicitly merges best video + best audio instead of a plain `-f mp4`, and supports both `--cookies-file` and `--cookies-from-browser`.
- `gesture_tracking.py` uses MediaPipe's current Tasks API (`HandLandmarker`), since the old `mp.solutions.hands` interface was removed from recent MediaPipe releases.
- `audio_prosody.py` detects F0 stress peaks only within contiguous voiced runs, avoiding a bug where unvoiced-gap zero-filling created spurious peaks at nearly every syllable boundary.
- `alignment.py` includes a permutation-based chance baseline (`permutation_test_match_rate`) so you can report a statistically meaningful match rate, not just a raw one.

## 1. Sanity-check your local environment

Run this first. If any of these fail, fix them before continuing — everything downstream depends on them.

In [1]:
import shutil, subprocess, sys

def check(name, cmd):
    path = shutil.which(cmd[0])
    if path is None:
        print(f"MISSING: {name} ('{cmd[0]}' not found on PATH)")
        return False
    try:
        out = subprocess.run(cmd, capture_output=True, text=True, timeout=10)
        first_line = (out.stdout or out.stderr).splitlines()[0] if (out.stdout or out.stderr) else ''
        print(f"OK: {name} -> {path}\n     {first_line}")
        return True
    except Exception as e:
        print(f"ERROR checking {name}: {e}")
        return False

ok_ffmpeg = check("ffmpeg", ["ffmpeg", "-version"])
ok_ytdlp = check("yt-dlp", ["yt-dlp", "--version"])
ok_js = any(shutil.which(x) for x in ("deno", "node", "bun"))
print(f"{'OK' if ok_js else 'MISSING'}: JS runtime for yt-dlp (deno/node/bun)")

if not (ok_ffmpeg and ok_ytdlp and ok_js):
    print("\n--> Fix the missing items above before continuing. See the markdown cell above for what's needed.")

OK: ffmpeg -> C:\Users\lenovo\AppData\Local\Microsoft\WinGet\Packages\Gyan.FFmpeg_Microsoft.Winget.Source_8wekyb3d8bbwe\ffmpeg-9.0.1-full_build\bin\ffmpeg.EXE
     ffmpeg version 9.0.1-full_build-www.gyan.dev Copyright (c) 2000-2026 the FFmpeg developers
OK: yt-dlp -> C:\Users\lenovo\AppData\Roaming\Python\Python313\Scripts\yt-dlp.EXE
     2026.08.19
OK: JS runtime for yt-dlp (deno/node/bun)


### 1b. If no JS runtime was found: install Deno

Skip this cell if the check above already showed Deno/Node/Bun as OK. On Windows, run this in **PowerShell** (not this notebook) since it needs to modify your user PATH, then restart your terminal and Jupyter kernel:

```powershell
irm https://deno.land/install.ps1 | iex
```

Then close and reopen PowerShell, reactivate your venv, restart this notebook's kernel, and re-run the check cell above.

## 2. (Optional) Configure cookies if YouTube blocks downloads with a bot-check

Only needed if step 3 below fails with `Sign in to confirm you're not a bot`. Since you're running locally, the most secure option is reading cookies directly from a logged-in browser — no file is written to disk. Set `COOKIES_FROM_BROWSER` below to `'chrome'`, `'firefox'`, `'edge'`, etc. (whichever browser you're logged into YouTube with), or leave both as `None` to skip cookies entirely and try without them first.

In [ ]:
COOKIES_FROM_BROWSER = None   # e.g. 'chrome', 'firefox', 'edge' -- set this if you hit the bot-check
COOKIES_FILE = None           # alternative: path to an exported cookies.txt (less secure, only if --cookies-from-browser doesn't work for your browser/setup)

## 3. Clean up any stale downloads, then list and download TED clips

If you've previously hit errors partway through (e.g. the ffmpeg-not-on-PATH crash), clear out any partial/unmerged files first.

In [2]:
import shutil as _shutil
from pathlib import Path

for d in (Path('data/raw'), Path('data/clips')):
    if d.exists():
        _shutil.rmtree(d)
    d.mkdir(parents=True, exist_ok=True)
print('Cleared data/raw and data/clips')

Cleared data/raw and data/clips


In [ ]:
!python src/download_clips.py --list-only

In [ ]:
import sys, subprocess

cmd = [sys.executable, 'src/download_clips.py', '--max-videos', '8', '--trim-start', '30', '--trim-duration', '60']
if COOKIES_FROM_BROWSER:
    cmd += ['--cookies-from-browser', COOKIES_FROM_BROWSER]
elif COOKIES_FILE:
    cmd += ['--cookies-file', COOKIES_FILE]

result = subprocess.run(cmd)
print(f"\nExit code: {result.returncode}")

### Sanity check: confirm a downloaded clip actually has an audio stream

Look for a `Stream #0:1: Audio:` line (in addition to the video stream) — if it's missing, the merge didn't happen (usually means ffmpeg wasn't on PATH during the download step).

In [ ]:
import glob
_clips = sorted(glob.glob('data/clips/*.mp4'))
if _clips:
    !ffprobe -hide_banner "{_clips[0]}"
else:
    print('No clips found yet -- check the download cell above for errors.')

## 4. Run the pipeline on each clip

In [9]:
import sys, glob, os
from pathlib import Path

# Walk up from cwd until we find a folder containing 'src' and 'notebook'
here = Path.cwd()
project_root = None
for candidate in [here] + list(here.parents):
    if (candidate / 'src').is_dir() and (candidate / 'notebook').is_dir():
        project_root = candidate
        break

if project_root is None:
    raise RuntimeError(f"Could not locate project root from cwd={here}. Check where Jupyter was launched from.")

print("Project root:", project_root)
sys.path.append(str(project_root / 'src'))
os.chdir(project_root)  # so relative paths like 'data/clips/*.mp4' work from here on

from audio_prosody import get_stress_peaks_for_video
from gesture_tracking import get_gesture_peaks_for_video
from alignment import match_peaks, summarize_offsets, plot_offset_histogram, plot_clip_timeline, permutation_test_match_rate

import numpy as np
import pandas as pd

clip_paths = sorted(glob.glob('data/clips/*.mp4'))
print(f"Found {len(clip_paths)} clips:")
for p in clip_paths:
    print(' -', p)

Project root: E:\cool_work\gesture-speech-alignment\gesture-speech-alignment
Found 8 clips:
 - data/clips\Clay_Shirky_-_How_cognitive_surplus_will_change_the_world_clip_30s-90s.mp4
 - data/clips\How_Arduino_is_open-sourcing_imagination_Massimo_Banzi_clip_30s-90s.mp4
 - data/clips\Jennifer_Pahlka_-_Coding_a_better_government_clip_30s-90s.mp4
 - data/clips\Jimmy_Wales_-_How_a_ragtag_band_created_Wikipedia_clip_30s-90s.mp4
 - data/clips\Massive-scale_online_collaboration_Luis_von_Ahn_clip_30s-90s.mp4
 - data/clips\The_Mind_Behind_Linux_Linus_Torvalds_TED_clip_30s-90s.mp4
 - data/clips\The_art_of_asking_Amanda_Palmer_clip_30s-90s.mp4
 - data/clips\Tim_Berners-Lee_-_The_year_open_data_went_worldwide_clip_30s-90s.mp4


In [10]:
import sys, glob
from pathlib import Path
sys.path.append('src')

from audio_prosody import get_stress_peaks_for_video
from gesture_tracking import get_gesture_peaks_for_video
from alignment import match_peaks, summarize_offsets, plot_offset_histogram, plot_clip_timeline, permutation_test_match_rate

import numpy as np
import pandas as pd

clip_paths = sorted(glob.glob('data/clips/*.mp4'))
print(f"Found {len(clip_paths)} clips:")
for p in clip_paths:
    print(' -', p)

Found 8 clips:
 - data/clips\Clay_Shirky_-_How_cognitive_surplus_will_change_the_world_clip_30s-90s.mp4
 - data/clips\How_Arduino_is_open-sourcing_imagination_Massimo_Banzi_clip_30s-90s.mp4
 - data/clips\Jennifer_Pahlka_-_Coding_a_better_government_clip_30s-90s.mp4
 - data/clips\Jimmy_Wales_-_How_a_ragtag_band_created_Wikipedia_clip_30s-90s.mp4
 - data/clips\Massive-scale_online_collaboration_Luis_von_Ahn_clip_30s-90s.mp4
 - data/clips\The_Mind_Behind_Linux_Linus_Torvalds_TED_clip_30s-90s.mp4
 - data/clips\The_art_of_asking_Amanda_Palmer_clip_30s-90s.mp4
 - data/clips\Tim_Berners-Lee_-_The_year_open_data_went_worldwide_clip_30s-90s.mp4


In [11]:
Path('results/example_timelines').mkdir(parents=True, exist_ok=True)
Path('data/tmp').mkdir(parents=True, exist_ok=True)

all_summaries = []
all_offsets_ms = []

for clip_path in clip_paths:
    clip_path = Path(clip_path)
    print(f"\nProcessing {clip_path.name} ...")

    stress_peaks, *_ = get_stress_peaks_for_video(clip_path)
    gesture_peaks, *_ = get_gesture_peaks_for_video(clip_path)

    print(f"  stress peaks: {len(stress_peaks)}, gesture peaks: {len(gesture_peaks)}")

    df = match_peaks(gesture_peaks, stress_peaks, window=0.5)

    # Chance-corrected significance test: is the match rate higher than
    # coincidental overlap given how dense the stress peaks are?
    perm = permutation_test_match_rate(gesture_peaks, stress_peaks, clip_duration=60.0, window=0.5, n_permutations=500)
    print(f"  observed match rate: {perm['observed_match_rate']:.2f}  |  chance match rate: {perm['null_mean_match_rate']:.2f} +/- {perm['null_std_match_rate']:.2f}  |  p={perm['p_value']:.3f}")

    summary = summarize_offsets(df)
    summary['chance_match_rate'] = perm['null_mean_match_rate']
    summary['permutation_p_value'] = perm['p_value']
    summary['clip'] = clip_path.name
    all_summaries.append(summary)

    matched_offsets = (df.loc[df['matched'], 'offset_s'] * 1000).values
    all_offsets_ms.extend(matched_offsets.tolist())

    out_path = f"results/example_timelines/{clip_path.stem}_timeline.png"
    plot_clip_timeline(clip_path.name, gesture_peaks, stress_peaks, out_path)
    print(f"  saved timeline -> {out_path}")

summary_df = pd.DataFrame(all_summaries)
summary_df


Processing Clay_Shirky_-_How_cognitive_surplus_will_change_the_world_clip_30s-90s.mp4 ...
  stress peaks: 263, gesture peaks: 22
  observed match rate: 1.00  |  chance match rate: 1.00 +/- 0.00  |  p=1.000
  saved timeline -> results/example_timelines/Clay_Shirky_-_How_cognitive_surplus_will_change_the_world_clip_30s-90s_timeline.png

Processing How_Arduino_is_open-sourcing_imagination_Massimo_Banzi_clip_30s-90s.mp4 ...
  stress peaks: 261, gesture peaks: 3
  observed match rate: 1.00  |  chance match rate: 1.00 +/- 0.00  |  p=1.000
  saved timeline -> results/example_timelines/How_Arduino_is_open-sourcing_imagination_Massimo_Banzi_clip_30s-90s_timeline.png

Processing Jennifer_Pahlka_-_Coding_a_better_government_clip_30s-90s.mp4 ...
  stress peaks: 270, gesture peaks: 19
  observed match rate: 1.00  |  chance match rate: 1.00 +/- 0.00  |  p=1.000
  saved timeline -> results/example_timelines/Jennifer_Pahlka_-_Coding_a_better_government_clip_30s-90s_timeline.png

Processing Jimmy_Wale

,n_gestures,n_matched,match_rate,median_offset_ms,mean_offset_ms,pct_gesture_precedes,chance_match_rate,permutation_p_value,clip
0,22,22,1.0,-25.000000,-31.363636,59.090909,1.000000,1.000,Clay_Shirky_-_How_cognitive_surplus_will_chang...
1,3,3,1.0,-50.000000,-33.333333,66.666667,1.000000,1.000,How_Arduino_is_open-sourcing_imagination_Massi...
2,19,19,1.0,-18.333333,-1.929825,57.894737,1.000000,1.000,Jennifer_Pahlka_-_Coding_a_better_government_c...
3,1,1,1.0,196.666667,196.666667,0.000000,1.000000,1.000,Jimmy_Wales_-_How_a_ragtag_band_created_Wikipe...
4,21,21,1.0,16.666667,0.952381,42.857143,0.993905,0.884,Massive-scale_online_collaboration_Luis_von_Ah...
5,17,17,1.0,26.666667,5.392157,41.176471,0.989647,0.830,The_Mind_Behind_Linux_Linus_Torvalds_TED_clip_...
6,15,15,1.0,-10.000000,-12.000000,53.333333,0.971600,0.714,The_art_of_asking_Amanda_Palmer_clip_30s-90s.mp4
7,11,11,1.0,9.666667,-29.930303,45.454545,1.000000,1.000,Tim_Berners-Lee_-_The_year_open_data_went_worl...


## 5. Aggregate results

In [12]:
summary_df.to_csv('results/per_clip_summary.csv', index=False)

all_offsets_ms = np.array(all_offsets_ms)
print(f"Total matched gesture-stress pairs across all clips: {len(all_offsets_ms)}")
if len(all_offsets_ms):
    print(f"Median offset: {np.median(all_offsets_ms):.1f} ms")
    print(f"Mean offset:   {np.mean(all_offsets_ms):.1f} ms")
    print(f"% gesture precedes stress: {(all_offsets_ms < 0).mean()*100:.1f}%")

    hist_path = plot_offset_histogram(all_offsets_ms)
    print(f"Saved histogram -> {hist_path}")
else:
    print("No matched pairs found -- try widening the matching window or checking peak detection thresholds.")

Total matched gesture-stress pairs across all clips: 109
Median offset: -10.0 ms
Mean offset:   -9.4 ms
% gesture precedes stress: 50.5%
Saved histogram -> results/offset_histogram.png


## 6. Results

Everything is already saved locally under `results/` — no download/zip step needed since you're running on your own machine. Open `results/offset_histogram.png`, `results/per_clip_summary.csv`, and `results/example_timelines/` directly in this project folder.